In [41]:
import pandas as pd

In [42]:
df = pd.read_csv("sales.csv")

In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [44]:
df.isna().sum().sum()

np.int64(6826)

In [45]:
df.dropna(inplace=True)

In [46]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9984,TXN_3142496,Smoothie,UNKNOWN,4.0,4.0,Cash,Takeaway,2023-07-27
9986,TXN_2858441,Sandwich,2,4.0,8.0,Credit Card,In-store,2023-12-14
9991,TXN_3897619,Sandwich,3,4.0,12.0,Cash,Takeaway,2023-02-24
9992,TXN_2739140,Smoothie,4,4.0,16.0,UNKNOWN,In-store,2023-07-05


In [47]:
df.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

In [48]:
df.duplicated().sum()

np.int64(0)

In [49]:
pd.Series(df['Transaction ID'].unique()).str.startswith("TXN").value_counts()

True    4550
Name: count, dtype: int64

### as we applu df["column_name].unique() -> we find that each column has some valuse as UNKNOWN and ERROR now we have to handle these missing values

The difference lies in what is being filtered: **row-filtering (dropping entire rows)** vs. **element-wise masking (replacing matching cells with `NaN`)**.

---

### 1. `df[~df.isin(["UNKNOWN", "ERROR"]).any(axis=1)]`

* **What it does:** **Filters and drops entire rows.**
* **How it works:**
1. `df.isin(...)` generates a boolean DataFrame of the same shape as `df`.
2. `.any(axis=1)` checks across columns for each row, returning a 1D Series of `True` (if the row has *at least one* `"UNKNOWN"` or `"ERROR"`) and `False`.
3. `~` inverts that Series (`True` becomes `False`).
4. Passing this 1D boolean Series into `df[...]` selects only the rows where **none** of those target values exist anywhere in that row.


* **Output Shape:** Same number of columns, but fewer (or equal) rows.
* **Cell Values:** Values remain completely unchanged.

---

### 2. `df[~df.isin(["UNKNOWN", "ERROR"])]`

* **What it does:** **Element-wise masking.**
* **How it works:**
1. `~df.isin(...)` produces a **2D boolean DataFrame** where cells matching `"UNKNOWN"` or `"ERROR"` are `False`, and all other cells are `True`.
2. When you pass a 2D boolean DataFrame of matching shape into `df[...]`, pandas keeps the cells that are `True` and turns any cell that is `False` into **`NaN`**.


* **Output Shape:** Exactly the same dimensions (rows and columns) as the original DataFrame.
* **Cell Values:** Any occurrence of `"UNKNOWN"` or `"ERROR"` is replaced by `NaN`. No rows are dropped.

---

### Quick Comparison

| Feature | `df[~df.isin(...).any(axis=1)]` | `df[~df.isin(...)]` |
| --- | --- | --- |
| **Action** | Drops matching rows entirely | Keeps all rows; masks matching cells |
| **Target Values Become** | Removed along with the row | Replaced with `NaN` |
| **Output Shape** | `(fewer_rows, same_cols)` | `(same_rows, same_cols)` |
| **Input to `df[...]**` | 1D Series (`[True, False, ...]`) | 2D DataFrame of booleans |

In [50]:
df.isin(["UNKNOWN", "ERROR"]).sum()

Transaction ID        0
Item                295
Quantity            146
Price Per Unit      168
Total Spent         155
Payment Method      346
Location            468
Transaction Date    128
dtype: int64

In [51]:
df.isin(["UNKNOWN"]).sum()

Transaction ID        0
Item                159
Quantity             75
Price Per Unit       80
Total Spent          75
Payment Method      167
Location            229
Transaction Date     60
dtype: int64

In [52]:
(df=="ERROR").sum()

Transaction ID        0
Item                136
Quantity             71
Price Per Unit       88
Total Spent          80
Payment Method      179
Location            239
Transaction Date     68
dtype: int64

In [53]:
df = df[~df.isin(["UNKNOWN","ERROR"]).any(axis=1)]

In [54]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
10,TXN_2548360,Salad,5,5.0,25.0,Cash,Takeaway,2023-11-07
12,TXN_7619095,Sandwich,2,4.0,8.0,Cash,In-store,2023-05-03
...,...,...,...,...,...,...,...,...
9975,TXN_9668108,Cake,1,3.0,3.0,Cash,In-store,2023-01-20
9979,TXN_9933628,Smoothie,5,4.0,20.0,Cash,In-store,2023-07-20
9986,TXN_2858441,Sandwich,2,4.0,8.0,Credit Card,In-store,2023-12-14
9991,TXN_3897619,Sandwich,3,4.0,12.0,Cash,Takeaway,2023-02-24


In [55]:
df.dtypes

Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

In [56]:
df['Quantity'] = df["Quantity"].astype(int)

In [57]:
df.dtypes

Transaction ID        str
Item                  str
Quantity            int64
Price Per Unit        str
Total Spent           str
Payment Method        str
Location              str
Transaction Date      str
dtype: object

In [58]:
df['Transaction Date'] = pd.to_datetime(df["Transaction Date"])

In [59]:
df.dtypes

Transaction ID                 str
Item                           str
Quantity                     int64
Price Per Unit                 str
Total Spent                    str
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

In [62]:
df = df.sort_values("Transaction Date")

In [63]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
8885,TXN_1581562,Coffee,2,2.0,4.0,Cash,In-store,2023-01-01
2726,TXN_2024598,Sandwich,1,4.0,4.0,Digital Wallet,In-store,2023-01-01
9356,TXN_2104473,Cake,3,3.0,9.0,Digital Wallet,Takeaway,2023-01-01
1806,TXN_2192787,Sandwich,5,4.0,20.0,Cash,In-store,2023-01-01
9063,TXN_9161256,Smoothie,2,4.0,8.0,Digital Wallet,In-store,2023-01-01
...,...,...,...,...,...,...,...,...
4017,TXN_6541110,Cookie,3,1.0,3.0,Cash,In-store,2023-12-30
2528,TXN_4592205,Cookie,3,1.0,3.0,Credit Card,In-store,2023-12-30
6515,TXN_7108012,Tea,1,1.5,1.5,Digital Wallet,Takeaway,2023-12-31
7441,TXN_1714348,Sandwich,2,4.0,8.0,Cash,Takeaway,2023-12-31


In [66]:
pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [67]:
df.to_excel("sales.xlsx")

In [68]:
pd.read_excel("sales.xlsx")

,Unnamed: 0,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,8885,TXN_1581562,Coffee,2,2.0,4.0,Cash,In-store,2023-01-01
1,2726,TXN_2024598,Sandwich,1,4.0,4.0,Digital Wallet,In-store,2023-01-01
2,9356,TXN_2104473,Cake,3,3.0,9.0,Digital Wallet,Takeaway,2023-01-01
3,1806,TXN_2192787,Sandwich,5,4.0,20.0,Cash,In-store,2023-01-01
4,9063,TXN_9161256,Smoothie,2,4.0,8.0,Digital Wallet,In-store,2023-01-01
...,...,...,...,...,...,...,...,...,...
3084,4017,TXN_6541110,Cookie,3,1.0,3.0,Cash,In-store,2023-12-30
3085,2528,TXN_4592205,Cookie,3,1.0,3.0,Credit Card,In-store,2023-12-30
3086,6515,TXN_7108012,Tea,1,1.5,1.5,Digital Wallet,Takeaway,2023-12-31
3087,7441,TXN_1714348,Sandwich,2,4.0,8.0,Cash,Takeaway,2023-12-31
